# Run this cell first

In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=MTH2100/numerical_analysis/interpolation"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except (AssertionError, ImportError):
      pass

# 3. Numerical Integration

## Why this matters

Many quantities in modelling are defined as integrals that cannot be computed analytically:
- Normalisation constants for probability distributions
- Partition functions in statistical mechanics
- Expected values, moments, and cumulative probabilities
- Areas under model prediction curves

We need efficient, accurate methods for evaluating $\displaystyle I = \int_a^b f(x)\,dx$.

**Learning objectives**
- Implement and understand the trapezoidal rule and Simpson's rule
- Understand and measure the error behaviour of each method
- Implement Monte Carlo integration and understand why it is sometimes preferred
- Apply these methods to problems arising in probability and statistical mechanics

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;">

**📹 VIDEO: Numerical Integration: Quadrature and Monte Carlo**

*What this video covers:* Why analytical integration often fails; the idea of numerical quadrature; trapezoidal and Simpson's rules and their error orders; Monte Carlo integration and its $O(1/\sqrt{N})$ convergence; when to use each approach.

*(Embed the video here using an `<iframe>` or `IPython.display.Video`)*
</div>

## 1. Quadrature Rules

A **quadrature rule** approximates $\int_a^b f(x)\,dx$ as a weighted sum of function
values at selected points (called *nodes*):
$$I \approx \sum_{k=0}^{n} w_k f(x_k)$$

The choice of nodes and weights determines the accuracy. The way the nodes and weights are constructed usually comes from approximating the integrand as a piecewise, interpolating polynomial, just like we looked at in the previous set of exercises. 

### Trapezium (or Trapezoidal) rule

You have probably seen this already at school: it's the easiest way to visualise the integral as the area under a curve. We approximate $f$ by a piecewise linear function on $n$ equal subintervals of width $h = (b-a)/n$. Then the area under each linear segment is a trapezium, and we can add all the individual areas up to get the integral. The formula is

$$I \approx \frac{h}{2}\!\left[f(x_0) + 2f(x_1) + 2f(x_2) + \cdots + 2f(x_{n-1}) + f(x_n)\right]$$

You can see the derivation [here](https://en.wikipedia.org/wiki/Trapezoidal_rule). The **Error: scales as $O(h^2)$** — i.e. halving $h$ reduces the error by a factor of 4.

### Simpson's rule

Here we approximate $f$ by a piecewise *quadratic* function on pairs of subintervals (note, this means $n$ must
be even). In other words, we take overlapping sets of three points, and find the quadratic functions which interpolate target function values, then integrate those quadratics. 

$$I \approx \frac{h}{3}\!\left[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 4f(x_{n-1}) + f(x_n)\right]$$

You can see the derivation of this formula [here](https://en.wikipedia.org/wiki/Simpson%27s_rule#Derivations): it's slightly less intuitive than the trapezium rule which is easy to visualise. Importantly though the **Error is prortional to $O(h^4)$** — halving $h$ reduces the error by a factor of 16. Thus, Simpson's rule is much more accurate than the trapezoidal rule for smooth functions.


---

To test these methods, let's calculate the value of an integral that can't be computed analytically, the Gaussian function. The integral we wish to compute is 

$$I = \int _{a} ^{b} e^{-x^2}dx,$$

which does not have a closed form antiderivative. Sure, if you look it up online you'll find that the integral can be given in terms of a special function called the Gauss error function ($\mathrm{erf}(x)$) but $\mathrm{erf}$ is defined in terms of an integral which itself has to be computed numerically. We'll pretend like it's the analytic solution for now and use it as our baseline for comprarison for the quadrature methods, but know that invoking $\mathrm{erf}$ is actually doing some hidden quadrature as well.

In [ ]:
import numpy as np
from scipy.special import erf

# ════════════════════════════════════════════════════════════════
# QUADRATURE IMPLEMENTATIONS
# ════════════════════════════════════════════════════════════════

def trapezoidal(f, a, b, n):
    '''
    Composite trapezoidal rule.

    Parameters
    ----------
    f    : callable — integrand
    a, b : floats   — integration limits
    n    : int      — number of subintervals (more = more accurate)

    Returns
    -------
    float — estimate of integral of f from a to b
    '''
    h = (b - a) / n                        # subinterval width
    x = np.linspace(a, b, n + 1)           # n+1 nodes
    y = f(x)                                # function values
    # Apply the composite trapezoidal formula: h/2 * [y_0 + 2*y_1 + ... + 2*y_{n-1} + y_n]
    return (h / 2) * (y[0] + 2 * np.sum(y[1:-1]) + y[-1])


def simpsons(f, a, b, n):
    '''
    Composite Simpson's rule (n must be even).

    Parameters
    ----------
    f    : callable — integrand
    a, b : floats   — integration limits
    n    : int      — number of subintervals (must be even)

    Returns
    -------
    float — estimate of integral of f from a to b
    '''
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's rule.")
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    # Weights: 1, 4, 2, 4, 2, ..., 4, 1
    weights = np.ones(n + 1)
    weights[1:-1:2] = 4   # odd-index interior nodes get weight 4
    weights[2:-2:2] = 2   # even-index interior nodes get weight 2
    return (h / 3) * np.dot(weights, y)


def f_demo(x):
    return np.exp(-x**2)   # Gaussian: exact integral from a to b is [erf(b) - erf(a) ]*sqrt(pi)/2

a, b = 0.0, 2.0

exact = (erf(2) - erf(0))*np.sqrt(np.pi)/2

NUMBER_OF_INTERVALS = 20

print(f"Trap rule integral      : {trapezoidal(f_demo, a, b, NUMBER_OF_INTERVALS)}")
print(f"Simpson's rule integral : {simpsons(f_demo, a, b, NUMBER_OF_INTERVALS)}")
print(f"'Analytic' integral     : {exact}")

Have a play with the `NUMBER_OF_INTERVALS` parameter above and see how the trapezium rule and Simpson's rule results change.

---


<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise
To see the $O(h^2)$ and $O(h^4)$ behaviour of the Trapezium and Simpson's rule integrals, we'll plot the error for a range of subsector sizes, $h$. For these methods, it's easier to work not with the step size but with the number of subsectors that we work with, but we'll plot vs $h$ because then we can see the scaling.

I've filled in how to plot the error for the trapezium method: you just have to add lines of code that will carry out the same steps for Simpson's rule.


</div>

In [ ]:
import matplotlib.pyplot as plt

trap_error = [] 

h_vals = []
N_vals = np.arange(10, 40, 2)

for n in N_vals:
    trap_error.append(exact - trapezoidal(f_demo, 0, 2, n))
    h_vals.append(2/n)

plt.semilogy(h_vals, trap_error, 'ro', label='Trapezium')

plt.xlabel('interval size, h')
plt.ylabel('Abs(Error)')
plt.legend()


# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

In [ ]:
runtest(['test_errors'])

Of course, in practice, we don't want to be implementing our own methods for doing quadrature: the above was just for example. The methods you'll need to compute integrals with quadrature are all part of the [`scipy.integrate`](https://docs.scipy.org/doc/scipy/reference/integrate.html) library. Oftentimes, as well, you will not have the function definition- above we defined the function `f_demo` and passed its _function handle_ (i.e. its name) as an argument to our functions, and those functions then computed the values of the function at a set of $x$-values. In practice, as we have discussed, you usually just have a list of numbers, e.g. from a measurement. The code snippet below loads some data (that happens to be the values of the same Gaussian function we were working with above), and computes the integral.

In [ ]:
from scipy.integrate import trapezoid, simpson

x, y = np.loadtxt('quad_data.txt')

trap_result = trapezoid(y, x)
simp_result = simpson(y, x)

print(trap_result)
print(simp_result)

In fact, the in-built `trapezoid` and `simpson` both _only_ work if you have a list of numbers. They don't work with function handles. So if you have the function you want to integrate (e.g. `f_demo`) and you want to use simpsons rule to integrate it, then you can define your own function to do that. 

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise
1. Define a function `simpson_func` which takes four arguments
   - `f` the name of the python function you want to integrate 
   - `a`, `b` the lower and upper limits for the definite integral
   - `N` the number of sub intervals for the quadrature
2. Inside `simpson_func`, define a numpy array `x` which contains the nodes (x-values) at which the function is to be evaluated. **Remember that the number of nodes is not the same as the number of sub-intervals!**
3. Inside `simpson_func`, use your array `x` and the function `f` which has been passed in to compute the array of function values, `y`
4. Now, use the built-in `simpson` function to compute the integral, and return that value

</div>


In [ ]:
runtest(['test_simp'])

Note that these methods presuppose uniform intervals. What that means is that if you have data that are not uniformly sampled (i.e. the gap between measurements is not the same each time) you can't just apply the Trapezium rule or Simpson's rule to those data. Thankfully, we now know methods for dealing with such data: remember the interpolation materials? So, given any non-uniformly sampled data, we simply need to construct an interpolating function (e.g. a cubic spline) and then use that for the integral. The data loaded in the following have randomly spaced points between 0 and 2. You should see that the result we get by just naively applying Simpson's rule to those data is incorrect, but by first applying the interpolation and then integrating with Simpson's rule, we get an answer that is pretty much spot on.

In [ ]:
from scipy.interpolate import make_interp_spline
import matplotlib.pyplot as plt

x_nu, y_nu = np.loadtxt('non_uniform_data.txt')

cs = make_interp_spline(x_nu, y_nu, k=3)
xx = np.linspace(0, 2, 101)
yy = cs(xx) 

plt.plot(x_nu, y_nu, 'r-', label='non-uniform data')
plt.plot(xx, yy, 'b--', label='uniform data')
plt.legend()

print(f"Integral with non-uniform data: {simpson(y_nu, x_nu)}")
print(f"Integral with uniform data    : {simpson(yy, xx)}")

# Note you can also use the function handle for the cubic spline if you're using a quadrature routine that accepts a function handle:

print(f"Integral using function handle: {simpson_func(cs, 0, 2, 100)}")

print(f"Exact integral for reference  : {exact}")


<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise
There are several other built-in integrators in the `scipy.integrate` library. One method called [_Gauss-Legendre quadrature_](https://en.wikipedia.org/wiki/Gauss–Legendre_quadrature) is extremely accurate and efficient. It can be accessed using the `scipy.integrate.fixed_quad` function as follows

```python
from scipy.integrate import fixed_quad

def function_example(x):
    ... # whatever function we want to integrate

I, _ = fixed_quad(function_example, a, b, n)
```
where `a` and `b` are the lower and upper limits and `n` is the number of nodes. 

We're going to compare the performance of the Gauss-Legendre quadrature with Simpson's rule

1. Define a function `fsin` which takes a single input argument `x` and returns the value of the function $\sin(\pi x^2)$
2. Use `fixed_quad` with 1001 nodes to compute the exact value of the integral, $\int _0 ^1 \sin (\pi x^2) dx$. Store that value as `I_exact`
3. Use a range of values for the number of nodes (up to, say, 20) and compute the integral using both `fixed_quad` and the `simpson_func` function you defined earlier. Remember that for Simpsons rule we supply the number of subintervals, whereas for `fixed_quad` it's the number of nodes. 
4. For each method, calculate the error as the absolute value of the difference between our computed integral and the exact integral. 
5. Plot the error as a function of the number of nodes for each method. 

Note that the tests only test the definition of `fsin` and the value of `I_exact`: the plot is not tested. If you do it right it should look something like this:

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

![](./GLvsSI.png)


</div>
</details>


</div>


In [ ]:
from scipy.integrate import fixed_quad




In [ ]:
runtest(['test_fsin', 'test_I_exact'])

## 2. Monte Carlo Integration

While quadrature is the most straightforward approach for calculating simple integrals, there are various reasons why it sometimes falls down. For instance, if we have a function of more than one variable, the number of function evaluations grows exponentially with the number of dimensions- this is called the 'curse of dimensionality'. 

An alternative approach is called 'Monte Carlo Integration' and it allows us to estimate definite integrals by taking the average of random function evaluations. You may have seen a common example of this before: [calculating the value of π by counting the number of uniform-randomly generated points in a square of side-length 2 which lie inside the unit circle](https://www.geeksforgeeks.org/dsa/estimating-value-pi-using-monte-carlo/). While that example is usually focused on calculating $\pi$, the method requires you to calculates an estimate for area, which is exactly what computing an integral is doing.

The key idea is to estimate $\int_a^b f(x)\,dx$ by drawing $N$ random points $x_1,\ldots,x_N$
uniformly from $[a,b]$ and averaging:

$$I \approx (b - a) \cdot \frac{1}{N} \sum_{k=1}^N f(x_k)$$

Geometrically, what this formula encodes is calculating the average value $\langle f \rangle$ of the function on the interval $(a, b)$, and then computing the area of the rectangle of side lengths $(b-a)$ and $\langle f \rangle$.

It turns out that this is a really inefficient way to integrate. The **convergence rate** is $O(1/\sqrt{N})$ — much slower than Simpson's rule which, recall, was $O(h^4)$, or $O(1/N^4)$ – for smooth functions in 1D. But in **high dimensions** ($d \gg 1$) quadrature rules require
exponentially many function evaluations ($O(n^d)$ nodes) whereas Monte Carlo still needs
only $O(N)$, independent of $d$. This makes Monte Carlo indispensable for
high-dimensional integration problems in statistics and statistical mechanics.



In [ ]:
# ════════════════════════════════════════════════════════════════
# MONTE CARLO INTEGRATION
# ════════════════════════════════════════════════════════════════

def monte_carlo_integrate(f, a, b, N):
    '''
    Estimate integral of f from a to b using N random samples.
    Returns (estimate, standard_error).
    '''
    from numpy.random import uniform
  
    x_samples = uniform(a, b, N)    # draw N uniform samples from [a, b]
    f_values  = f(x_samples)            # evaluate f at each sample
    estimate  = (b - a) * np.mean(f_values)   # scale by interval width
    std_err   = (b - a) * np.std(f_values) / np.sqrt(N)  # standard error
    return estimate, std_err

N_POINTS = 100

integral, error = monte_carlo_integrate(f_demo, 0, 2, N_POINTS)

print(f"MC estimate   : {integral} ± {error}")
print(f"Exact integral: {exact}")

Now, if you run the code cell above multiple times, you'll find you get a different answer each time: that's to be expected, because we're using random numbers. You'll also notice that a lot of times, the answer is pretty rubbish. Again, that's to be expected, because we're only using a small number of points (100). You can partially address that by increasing the value of `N_POINTS`- you should see the error becoming smaller. We can also improve things by doing the simulation several times, and taking the average of the result. This also lowers the chance that a single bad result ruins our lives- on average things should look a little more reasonable. 

In [ ]:
N_ESTIMATES = 10

running_total = 0
for estimate in range(N_ESTIMATES):
    integral, error = monte_carlo_integrate(f_demo, 0, 2, N_POINTS)
    running_total = running_total + integral

integral = running_total / N_ESTIMATES

print(f"MC estimate   : {integral}")
print(f"Exact integral: {exact}")

You should notice that the values for the integral provided by this method are much closer to the exact value of the integral. There's a lot more to learn about Monte-Carlo methods, including (very importantly) how to estimate errors, but you will find these in other exercises

# ADD LINKS TO GARETHS MC EXERCISES

By the way, as with our 'self-defined' `trapezoidal` and `simpsons` functions, there is a built in function for performing Monte-Carlo integration, and in fact it does multiple simulations and averages all at once so that you don't need to do it yourself. The function is `qmc_quad` (quasi-monte carlo quadrature). The number of points (the sample size) is supplied as the argument `n_points`, and the number of simulations that are averaged to give the final result is `n_estimates`. Just like our self-defined `monte_carlo_integrate` function, `qmc_quad` returns both the value of the integral, and the standard error:

In [ ]:
from scipy.integrate import qmc_quad

Integral, error = qmc_quad(f_demo, 0, 2, n_estimates = 8, n_points = 1024)
print(Integral, error)